In [14]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "allritz2013food")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "apples_reliability.sav")
complete_path_2 = os.path.join(original_data_pathway, "chocopops_reliability.sav")
complete_path_3 = os.path.join(original_data_pathway, "raw_data_cleaned.sav")


out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [15]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'apples_reliability.csv')
# df1.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'chocopops_reliability.csv')
# df2.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

df3 = pd.read_spss(complete_path_3, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'raw_data_cleaned.csv')
# df3.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)

In [16]:

df3['study_id']="allritz2013food"
df3.columns = map(str.lower, df3.columns)
df3=df3.applymap(lambda s: s.lower() if type(s) == str else s)
df3.columns
df3.rename(columns={"animal": "participant",
                    "species":"species_original",
                    "age":"age_original",
                    "age_grp":"age_group",
                    "chimp_grp":"species_subgroup",
                    "sex":"sex_original",
                    "fo_wa_report":"food_washing_reports_category",
                    "fo_wa_mother":"food_washing_mother",
                    "rnd_grp_n":"random_group_number",
                    "exp_grp":"experimental_group",
                    "ord_food":"food_order",
                    "ord_state":"state_order",
                    "date_m":"month",
                    "date_d":"day",
                    "c_was":"c_in_water",
                    "c_eat":"c_eaten",
                    "c_was_e":"c_in_water_then_eaten",
                    "a_obt":"a_obtained",
                    "a_cle":"a_cleaned",
                    "a_eat":"a_eaten",
                    "a_was":"a_in_water",
                    "a_was_w":"whole_a_in_water",
                    "a_was_p":"a_partially_washed",
                    "a_was_e":"a_washed_then_eaten",
                    "a_was_w_e":"whole_a_washed_then_eaten",
                    "a_was_rr":"retrieve-release_occured",
                    "a_was_d":"dunking_occured",
                    "a_was_all":"always_washed_before_eating",
                    "a_was_ins":"inspected_between_washing_and_eating",
                    "a_was_pla":"not_eaten_after_washing",
                    "a_was_w_not_e":"whole_a_washed_but_not_eaten",
                    "c_was_not_e":"c_washed_but_not_eaten"}, inplace=True)

df3.columns = df3.columns.str.replace('c_', 'chocopops_')
df3.columns = df3.columns.str.replace('a_', 'apples_')


In [17]:
nan_replace = ['day','month','chocopops_eaten','chocopops_in_water',
       'chocopops_in_water_then_eaten']
for x in nan_replace:
    df3[x].replace(99., np.nan, inplace=True, regex=True)
# df3['month'].unique()


In [18]:

year_list_1 = [ 10.,12., 11., 8.,  9.,  7.]
year_list_2 = [  1.,  2.]


for x in year_list_1:
    df3.loc[df3.month == x, ['year']] = '2010'
for x in year_list_2:
    df3.loc[df3.month == x, ['year']] = '2011'



In [19]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df3['participant'] = df3['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df3['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df3= df3.merge(apedf,left_on='participant', right_on='name', how='left')


In [20]:
df3['month']=df3['month'].astype("Int64")
df3['day']=df3['day'].astype("Int64")

# df3['month'].unique()

In [21]:
# df3['month']=df3['month'].str.rstrip('.0')
# df3['day']=df3['day'].astype(str)
# df3['day']=df3['day'].str.rstrip('.0')
# df3['day'].unique()



In [22]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df3= df3.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df3['dodc'] = df3['year'].astype(str) + '-' + df3['month'].astype(str) + '-' + df3['day'].astype(str)
df3['dodc'].replace('nan-nan-nan', np.nan, inplace=True, regex=True)
df3['dodc'].replace('nan-<NA>-<NA>', np.nan, inplace=True, regex=True)
df3['dodc'] = pd.to_datetime(df3['dodc'])
df3['dob'] = pd.to_datetime(df3['dob'])

df3['age_in_years'] = (df3['dodc'] - df3['dob']).dt.days//365 

df3=df3.sort_values(by = ['participant','condition','trial'])


In [23]:
# df3['month'].replace('nan', np.nan, inplace=True, regex=True)
# df3['day'].replace('nan', np.nan, inplace=True, regex=True)
# df3['month'].unique()


In [24]:
import re
replace_1=re.compile('(\/|\,|\?)')
df3['notes'].replace(replace_1, '', inplace=True, regex=True)
df3['notes'].replace('"jahaga"', 'jahaga', inplace=True, regex=True)
df3['notes'].replace('"washed"', 'washed', inplace=True, regex=True)

df3['experimental_group'].replace(' ', '', inplace=True, regex=True)

space_list= ['condition','food_order','state_order','notes']
for x in space_list:
    df3[x].replace(' ', '_', inplace=True, regex=True)


In [25]:
rank_rename = [[1,'high'],
                [2,'medium'],
                [3,'low']]
for x,y in rank_rename:
    df3['rank'].replace(x, y, inplace=True, regex=True)
# df3.columns

df3=df3.sort_values(by = ['participant', 'year','month','day'])

In [26]:


studyID_standardized=df3[['study_id', 'year','month',  'day', 
                          'participant', 'age_original','age_in_years', 'age_group','sex', 'species',
       'species_subgroup', 'trial','condition', 'food_order',
       'state_order', 'rank',
       'food_washing_reports_category', 'food_washing_mother',
       'random_group_number', 'experimental_group', 
       'chocopops_eaten', 'chocopops_in_water',
       'chocopops_in_water_then_eaten', 'apples_obtained', 'apples_eaten',
       'apples_cleaned', 'apples_in_water', 'whole_apples_in_water',
       'apples_partially_washed', 'apples_washed_then_eaten',
       'whole_apples_washed_then_eaten', 'retrieve-release_occured',
       'dunking_occured', 'always_washed_before_eating',
       'inspected_between_washing_and_eating', 'not_eaten_after_washing',
       'whole_apples_washed_but_not_eaten',
       'chocopops_washed_but_not_eaten' 
       ]]
comp_out_path_stand = os.path.join(out_pathway, 'allritz2013food_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'allritz2013food_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
